<a href="https://www.kaggle.com/code/mariopaerle/modern-llms-model-with-vathos?scriptVersionId=317473968" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, IterableDataset
from datasets import load_dataset
from transformers import AutoTokenizer

class FineWebStreamingDataset(IterableDataset):
    def __init__(self, hf_dataset, tokenizer, max_length=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __iter__(self):
        for item in self.dataset:
            text = item.get('text', '')
            if not text or len(text) < 10: 
                continue
            
            tokenized = self.tokenizer(
                text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            
            yield {
                'input_ids': tokenized['input_ids'].squeeze(0),
                'attention_mask': tokenized['attention_mask'].squeeze(0)
            }

def get_dataloaders(model_name="gpt2", batch_size_train=6, batch_size_val=4, max_length=512, val_size=1000):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True)
    shuffled_ds = ds.shuffle(seed=42, buffer_size=10000)
    
    train_raw = shuffled_ds.skip(0)
    val_raw   = shuffled_ds.skip(9_000_000).take(val_size)
    
    train_dataset = FineWebStreamingDataset(train_raw, tokenizer, max_length=max_length)
    val_dataset = FineWebStreamingDataset(val_raw, tokenizer, max_length=max_length)
    train_loader = DataLoader(train_dataset, batch_size=batch_size_train, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size_val)
    
    return train_loader, val_loader, tokenizer

In [10]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
VAL_SIZE = 100
train_loader, val_loader, tokenizer = get_dataloaders(
    model_name="gpt2", 
    batch_size_train=32, 
    batch_size_val=4, 
    max_length=256,
    val_size=VAL_SIZE
)

TOKEN_PAD = tokenizer.pad_token_id
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Padding ID: {TOKEN_PAD}")

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Tokenizer vocab size: 50257
Padding ID: 50256


In [ ]:
result_dict = {}

In [4]:
!pip uninstall aplos -y
!pip install -q git+https://github.com/MarioPaerle/Aplos@VathosUI
from Vathos.blocks import *

In [ ]:
!pip install git+https://github.com/KellerJordan/Muon.git

In [6]:
device = 'cuda'
d_model = 512
n_layers = 20
expands = [4 for i in range(n_layers)]
d_models = [d_model for i, e in enumerate(expands)]
m_dims = [d_model * e for i, e in enumerate(expands)]

attn = Builder(Attention, n_heads=8, pos_emb=RoPE(d_model//8), causal=True, qk_norm=True)
attns = [attn for i in range(n_layers)]

model = ModdedFormer(
    vocab_size=50304,
    embed_dim=d_model,
    d_models=d_models,
    spatials=attns,
    M_dims=m_dims, 
    ffn_act=LeakyReLU2,
    zeroskip=True,
    baseblock=Block1d,
    UDLP=VariableUDLP,
    weights_tying=True 
).to(device)

model.name = "008"
model.summary()
model._init_weights()
model = nn.DataParallel(model) # Simple dataparallelism across two T4s, maybe slower then a single one, but enables a bigger batch size

Vathos ModdedFormer Summary
Embedding dim: 512
Learnable PE: False
Skips: [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
Layer 0: 512 -> 512
Block: 0: Vathos: Block1d(
  (spatial_mixer): Vathos: MultiheadAttentionMixer(
    (qkv): Linear(in_features=512, out_features=1536, bias=False)
    (out): Linear(in_features=512, out_features=512, bias=False)
    (pos_emb): Vathos: RoPE()
    (q_norm): RMSNorm()
    (k_norm): RMSNorm()
  )
  (channel_mixer): Vathos: VariableUDLP(
    (expand): Linear(in_features=512, out_features=2048, bias=False)
    (contract): Linear(in_features=2048, out_features=512, bias=False)
    (activation): Vathos: LeakyReLU2()
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (norm1): RMSNorm()
  (norm2): RMSNorm()
)
	FFN Dimension: 2048
	FFN Activation: Vathos: LeakyReLU2()
Layer 1: 512 -> 512
Block: 1: Vathos: Block1d(
  (spatial_mixer): Vathos: MultiheadAttentionMixer(
    (qkv): Linear(in_f

In [7]:
from muon import MuonWithAuxAdam
from torch.optim import AdamW

def trapezoidal_lr(step,
                   warmup_steps,
                   plateau_steps,
                   total_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    elif step < warmup_steps + plateau_steps:
        return 1.0
    else:
        decay_steps = total_steps - warmup_steps - plateau_steps
        step_in_decay = step - warmup_steps - plateau_steps
        return max(0.0, 1.0 - step_in_decay / max(1, decay_steps))

hidden_weights = []
hidden_gains_biases = []
nonhidden_params = [] 

embed_and_head_keywords = ['embedding', 'unembedder', 'embedder', 'unembed', 'output']

for name, p in model.named_parameters():
    if any(keyword in name for keyword in embed_and_head_keywords):
        print(name)
        nonhidden_params.append(p)
    elif p.ndim >= 2:
        hidden_weights.append(p)
    else:
        hidden_gains_biases.append(p)

param_groups = [
    dict(
        params = hidden_weights,
        use_muon = True,
        lr = 1e-2,        
        weight_decay = 0.01
    ),
    dict(
        params = hidden_gains_biases + nonhidden_params,
        use_muon = False,
        lr = 5e-3,          
        betas = (0.9, 0.999), 
        weight_decay = 0.01
    ),
]

optimizer = MuonWithAuxAdam(param_groups)
# optimizer = SingleDeviceStochasticSignMuon(param_groups)
# optimizer = StochasticMuonWithAuxAdam(param_groups, 5000)
# optimizer = HybridMuonWithAuxAdam(param_groups)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: trapezoidal_lr(
        step,
        warmup_steps=250,
        plateau_steps=15_000, # 10_000
        total_steps=32_000 # 15000
    )
)

module.embedder.embedding.weight


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_scheduler(scheduler_fn, total_steps, title="Learning Rate Schedule"):
    steps = np.arange(total_steps)
    lrs = [scheduler_fn(step) for step in steps]
    
    plt.figure(figsize=(10, 5))
    plt.plot(steps, lrs, linewidth=2)
    plt.xlabel("Steps")
    plt.ylabel("LR Multiplier")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

# Usage:
plot_scheduler(
    lambda step: trapezoidal_lr(step, warmup_steps=1_000, plateau_steps=15_000, total_steps=32_000),
    total_steps=32_000
)

In [8]:
import torch.distributed as dist

dist.init_process_group(
    backend='nccl',
    init_method='tcp://127.0.0.1:29500',
    world_size=1,
    rank=0
)

In [ ]:
from tqdm.notebook import tqdm
import torch
import torch.nn.functional as F
from timeit import default_timer as timer

set_vathos_mode('production')
scaler = torch.amp.GradScaler("cuda") 

model.train()
SPE = 1000 
num_epochs = 20
acc_step = 1
finished = False
finished1 = False
finished2 = False

seq_len = 32
step = 0

train_iter = iter(train_loader)
optimizer.zero_grad(set_to_none=True)

global_batch_step = 0
loss_history = []
loss_history_timed = {}
ms_history = []  
ema_ms = None    
start_time = timer()
stop_training_reached_treshold = False
print("Timer Started")

for epoch in range(num_epochs):
    total_loss = 0
    pbar = tqdm(range(SPE), desc=f"Epoch {epoch+1}/{num_epochs}")
      
    if finished:
        break
        
    for count in pbar:
        step_start_time = timer() 
        
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch['input_ids'].to(device)
        
        x = input_ids[:, :seq_len-1].contiguous()
        y = input_ids[:, 1:seq_len].contiguous()

        
        with torch.amp.autocast("cuda"):
            pred = model(x)
            B, T, V = pred.shape
            loss = F.cross_entropy(
                pred.view(-1, V), 
                y.view(-1), 
                ignore_index=tokenizer.pad_token_id
            )
            loss = loss / acc_step 

        scaler.scale(loss).backward()
        
        global_batch_step += 1
        
        if global_batch_step % acc_step == 0:
            
            # scaler.unscale_(optimizer)
            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            
            old_scale = scaler.get_scale()
            scaler.update()
            new_scale = scaler.get_scale()
            
            if new_scale >= old_scale:
                scheduler.step()
                
            optimizer.zero_grad(set_to_none=True)
        
        if step % 500 == 0:
            seq_len = min(seq_len*2, 256)
            acc_step = min(acc_step + 1, 5)
        step += 1
    

        ########################################################################### NON MODIFICABILE
        current_lr = optimizer.param_groups[0]['lr']
        loss_val = loss.item() * acc_step
        total_loss += loss_val
        

        k = 50
        loss_history.append(loss_val)
        if len(loss_history) > k:
            loss_history.pop(0)
            
        n = len(loss_history)
        weights = [i + 1 for i in range(n)]
        weighted_loss = sum(w * l for w, l in zip(weights, loss_history)) / sum(weights)
        if stop_training_reached_treshold:
            if weighted_loss <= 5 and not finished1:
                print(f"5.0 CE in {timer() - start_time:.2f}s and {count + epoch*SPE} steps")
                finished1 = True
            elif  weighted_loss <= 4.5 and not finished2:
                 print(f"4.5 CE in {timer() - start_time:.2f}s and {count + epoch*SPE} steps")
                 finished2 = True
            elif weighted_loss <= 4:
                print(f"Finish! 4.0 in {timer() - start_time:.2f}s and {count + epoch*SPE} steps")
                finished = True
                break

        _elapsed = timer() - start_time
        loss_history_timed[_elapsed] = loss_val


        step_ms = (timer() - step_start_time) * 1000 
        if ema_ms is None:
            ema_ms = step_ms
        else:
            ema_ms = 0.1 * step_ms + 0.9 * ema_ms 
        
        ms_history.append(ema_ms)

        pbar.set_postfix({
            'avg_loss': f"{total_loss/(count+1):.4f}", 
            'lr': f"{current_lr:.2e}",
            f'MA@{k}': weighted_loss,
            'ms/step': f"{ema_ms:.1f}"  
        })

        if hasattr(model, 'module'):
            model.module.register_loss(loss_val)
        else:
            model.register_loss(loss_val)
            
    if not finished and (global_batch_step % acc_step != 0):
        scaler.step(optimizer)
        old_scale = scaler.get_scale()
        scaler.update()
        if scaler.get_scale() >= old_scale:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)
    
    if hasattr(model, 'module'):
        model.module.register_epoch()
    else:
        model.register_epoch()
    print(f"Elapsed time: {timer() - start_time}s")

model.save_checkpoint(f"{model.name}.pt")

Vathos: Switched to PRODUCTION mode.
Timer Started


Epoch 1/20:   0%|          | 0/1000 [00:00<?, ?it/s]

Elapsed time: 426.3570973550004s


Epoch 2/20:   0%|          | 0/1000 [00:00<?, ?it/s]

Elapsed time: 1019.5107645440003s


Epoch 3/20:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
from Vathos.Analyzer import study

analyzer = study(model, layer_idx=4)

In [ ]:
model.plot_losses()

In [ ]:
from Vathos.functions import simple_ar_generate                                                                                                                                     

prompt = "Hemogoblin is"
prompti = tokenizer(prompt, return_tensors="pt")['input_ids']
print(prompti)
out = simple_ar_generate(
      model, prompti,                       # prompt: tensor [L] o [B,L]                                                                                                               
      max_len=100,                                          
      temperature=0.5, top_k=50, top_p=0.9,
      repetition_penalty=1.12,
      token_end=None,                      # opz. id di stop
  )

print(tokenizer.decode(out[0]))
out = model.generate(
      prompti,
      max_len=100,
      temperature=1.0, top_k=50, top_p=0.9,
      repetition_penalty=1.1,
      token_end=None,
      simple=False,                        # True ⇒ fallback al naive (oracolo di test)
  )
print(tokenizer.decode(out[0]))
